In [3]:
from nrem_sc.constants import RAW_DATA_PATH, PROCESSED_DATA_PATH, SAMPLING_RATE
from nrem_sc.utils import group_by_ids

from scipy.io import loadmat
import numpy as np
import pandas as pd
import pynapple as nap

unit_id = '119b'

def group_to_tsgroup(values: np.ndarray, ids: np.ndarray, select_ids=None) -> nap.TsGroup:
    grouped = group_by_ids(values, ids, select_ids)
    return nap.TsGroup({uid: nap.Ts(spikes, time_units='s') for uid, spikes in grouped.items()})

### Head Direction angle

In [ ]:
print(f"Processing unit {unit_id}...")

t = np.load(RAW_DATA_PATH / unit_id / f"YutaTest{unit_id}_t_1ms_OpenField").squeeze()
print(f"Loaded time: {t.shape} and range {t[0]} - {t[-1]}")
hd_angle = np.load(RAW_DATA_PATH / unit_id / f"YutaTest{unit_id}_angle_all_1ms_OpenField").squeeze()
print(f"Loaded head direction angle: {hd_angle.shape} and range {hd_angle.min()} - {hd_angle.max()}")
nap.Tsd(t, hd_angle).save(PROCESSED_DATA_PATH / unit_id / f"angle_openfield.npz")
print(f"Saving to: {PROCESSED_DATA_PATH / unit_id / f'angle_openfield.npz'}")

### HD cells

In [ ]:
# Extraction of HD cell ids in Mouse 107b
hd_ids = loadmat(PROCESSED_DATA_PATH / "hd_ids.mat")['hd_ids'].squeeze()
hd_ids_1, hd_ids_2 = np.split(hd_ids, np.where(np.diff(hd_ids) >= 30)[0] + 1) # Split to shank 1 and 2 ids
hd_ids

In [21]:
hd_spikes = []
path = RAW_DATA_PATH / unit_id
number_of_shanks = len(list(path.glob('*clu*')))
unit_idx = loadmat(PROCESSED_DATA_PATH / unit_id /'unit_idx.mat')['unit_idx'].squeeze()
total = 0

print(f"Reading from {RAW_DATA_PATH / unit_id}...")
print(f"Found {number_of_shanks} shanks with {len(unit_idx)} clusters and {sum(unit_idx)} HD cells.")
for i in range(1, number_of_shanks + 1):
    print(f"Reading Shank {i}...")
    # Use np.loadtxt for 83b
    clu = np.loadtxt(RAW_DATA_PATH / unit_id / f"YutaTest{unit_id}.clu.{i}").squeeze()
    res = np.loadtxt(RAW_DATA_PATH / unit_id / f"YutaTest{unit_id}.res.{i}").squeeze()
    print(f"Shank {i}: Loaded cluster identities with shape {clu.shape} and spike times with shape {res.shape}")
    
    clu = clu[1:]  # remove number of clusters at the beginning
    units = np.unique(clu).astype(int)
    units = units[(units != 0) & (units != 1)] # exclude noise and multiunit clusters
    number_of_clusters = len(units)
    shank_hd = unit_idx[total:total + number_of_clusters]
    hd_ids = units[shank_hd.astype(bool)]

    print(f"Shank {i}: {int(number_of_clusters)} clusters with IDs: {units}")

    if len(hd_ids) == 0:
        print(f"Shank {i}: No HD cells found, skipping...")
        continue
    else:
        print(f"Shank {i}: {sum(shank_hd)} HD cells with IDs: {hd_ids}")

    hd_spikes.append(group_to_tsgroup(values=res/SAMPLING_RATE,
                                 ids=clu,
                                 select_ids=hd_ids))
    total += number_of_clusters
    assert len(hd_spikes[-1]) == len(hd_ids)

hd_1 = hd_spikes[0]
hd = hd_1.merge(*hd_spikes[1:], reset_index=True, reset_time_support=True, ignore_metadata=True)
hd.save(PROCESSED_DATA_PATH / unit_id / "hd_spikes_total")


Reading from D:\common_datasets\ucsf\raw\83b...
Found 2 shanks with 86 clusters and 32 HD cells.
Reading Shank 1...
Shank 1: Loaded cluster identities with shape (11291718,) and spike times with shape (11291717,)
Shank 1: 27 clusters with IDs: [ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25
 26 27 28]
Shank 1: 20 HD cells with IDs: [ 2  4  5  6  7  8  9 10 11 12 13 14 18 20 21 22 24 26 27 28]
Reading Shank 2...
Shank 2: Loaded cluster identities with shape (18399517,) and spike times with shape (18399516,)
Shank 2: 59 clusters with IDs: [ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25
 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49
 50 51 52 53 54 55 56 57 58 59 60]
Shank 2: 12 HD cells with IDs: [ 8 10 20 23 30 32 40 42 49 51 53 58]


### Sleep States

In [44]:
sleep_df = []
# states = ['wake', 'nrem', 'rem']
# 85b
unit_id = '119b'
states = ['wake', 'nrem', 'rem']
for state in states:
    print(f"Loading: {RAW_DATA_PATH / unit_id / f'{state}.mat'}...")
    data = loadmat(RAW_DATA_PATH / unit_id / f"{state}.mat")[state]
    sleep_df.append(pd.DataFrame({'start': data[:,0], 'end': data[:,1], 'state': state}))

sleep_df = pd.concat(sleep_df).sort_values('start').reset_index(drop=True)
sleep_df['start'] = sleep_df['start'] - 1 # MATLAB indices to Python indices
# Drop rows where start and end values are equal
sleep_df = sleep_df[sleep_df.iloc[:, 0] != sleep_df.iloc[:, 1]]
sleep = nap.IntervalSet(start=sleep_df['start'], end=sleep_df['end'], time_units='s', metadata={'state' : sleep_df['state'].values})
# sleep.save(PROCESSED_DATA_PATH / unit_id / "sleep")

Loading: D:\common_datasets\ucsf\raw\119b\wake.mat...
Loading: D:\common_datasets\ucsf\raw\119b\nrem.mat...
Loading: D:\common_datasets\ucsf\raw\119b\rem.mat...


C:\Users\iii9781\AppData\Local\Temp\ipykernel_5340\2618467586.py:15: UserWarning: Some starts and ends are equal. Removing 1 microsecond!
  sleep = nap.IntervalSet(start=sleep_df['start'], end=sleep_df['end'], time_units='s', metadata={'state' : sleep_df['state'].values})


### TTX injection data

In [12]:
unit_id = '119b'
s = loadmat(RAW_DATA_PATH / unit_id / "session_epochs.mat", squeeze_me=True)['session_epochs']
labels = ['homecage', 'openfield', 'ttx', 'homecage']

# Concatenate last three homecage sessions
s_new = s[:4]
s_new[-1, 1] = s[-1, 1]
nap.IntervalSet(start=s_new[:, 0], end=s_new[:, 1], time_units='s', metadata={'label': labels}).save(PROCESSED_DATA_PATH / unit_id / "sessions_labeled")

In [ ]:
unit_id = '85b'
s = loadmat(PROCESSED_DATA_PATH / unit_id / "session_epochs.mat", squeeze_me=True)['session_epochs']
labels = ['openfield', 'homecage', 'ttx', 'openfield', 'homecage', 'openfield']
nap.IntervalSet(start=s[:, 0], end=s[:, 1], time_units='s', metadata={'label': labels}).save(PROCESSED_DATA_PATH / unit_id / "sessions_labeled")

In [42]:
unit_id = '83b'
s = loadmat(RAW_DATA_PATH / unit_id / "session_epochs.mat", squeeze_me=True)['session_epochs']
labels = ['openfield', 'homecage', 'ttx', 'openfield', 'homecage']

# Connect first two consecutive homecage sessions
s_new = np.vstack([s[0], s[2:]])
s_new[1, 0] = s_new[0, 1] + 0.00005

nap.IntervalSet(start=s_new[:, 0], end=s_new[:, 1], time_units='s', metadata={'label': labels}).save(PROCESSED_DATA_PATH / unit_id / "sessions_labeled")

### ADN-SC recording data

In [ ]:
p = loadmat(PROCESSED_DATA_PATH / "neck_pos.mat")['pos']
t = loadmat(PROCESSED_DATA_PATH / "neck_time.mat")['t_new'].squeeze()
pos = nap.TsdFrame(t=t, d=p, columns=['x', 'y'])
pos.save(PROCESSED_DATA_PATH / "position_neck")

speed = np.linalg.norm(pos.derivative(), axis=1)
thr = np.quantile(speed, 0.25)

active_period = speed.threshold(thr).time_support
print(f"Total active duration: {active_period.tot_length()} s")

wake = sleep[sleep.groupby('state')['wake']]
active_wake = wake.intersect(active_period)
print(f"Active wake duration: {active_wake.tot_length()} s")
active_wake.save(PROCESSED_DATA_PATH / "active_wake")
active_wake


In [ ]:
# turn_ids contains the indices of the cluster ids
turn_ids = loadmat(PROCESSED_DATA_PATH / unit_id / "turn_ids.mat")['turn_ids'].squeeze()
# ccw_ids = loadmat(PROCESSED_DATA_PATH / unit_id /  "id_ccw.mat")['id_ccw'].squeeze()
mod_idx = loadmat(PROCESSED_DATA_PATH / unit_id /  "mod_idx.mat")['mod_idx'].squeeze()
print(turn_ids)
print(mod_idx)
assert len(turn_ids) == len(mod_idx)

sc_spikes = nap.load_file(PROCESSED_DATA_PATH / unit_id / "spikes_shank_3.npz")
sc_spikes


# turn ids are 1-indexed, so we need to subtract 1 to get the correct index in sc_spikes
sc_turn_ids = [sc_spikes.index[tid - 1] for tid in turn_ids]

# turn_spikes = nap.TsGroup(
#     {sid: sc_spikes[sid] for sid in sc_turn_ids},
#     metadata={"turn_mode": np.where(ccw_ids, "ccw", "cw")}
# )

turn_spikes = nap.TsGroup(
    {sid: sc_spikes[sid] for sid in sc_turn_ids},
    metadata={"cw_modulation": mod_idx[:, 0], 'ccw_modulation': mod_idx[:, 1]}
)

turn_spikes

turn_spikes.save(PROCESSED_DATA_PATH / unit_id / "turn_spikes")
